## Checking correctness 

In [0]:
%sql
--- Checking distinct IDs in fct_result ---
SELECT
    COUNT(DISTINCT athlete_id) AS unique_athlete_id,
    COUNT(DISTINCT event_id) AS unique_event_id,
    COUNT(DISTINCT date_id) AS unique_date_id
FROM marathos_cat.gold.fct_result;

In [0]:
%sql
--- Checking row count in dim tables ---
SELECT 'dim_athlete' AS table_name, COUNT(*) AS rows FROM marathos_cat.gold.dim_athlete
UNION ALL
SELECT 'dim_event', COUNT(*) FROM marathos_cat.gold.dim_event
UNION ALL
SELECT 'dim_date', COUNT(*) FROM marathos_cat.gold.dim_date
UNION ALL
SELECT 'fct_result', COUNT(*) FROM marathos_cat.gold.fct_result;

In [0]:
%sql
--- Checking dim_athlete ---
SELECT * FROM marathos_cat.gold.dim_athlete LIMIT 5;

In [0]:
%sql
SELECT
    COUNT(DISTINCT athlete_id) AS unique_athletes_in_fct,
    (SELECT COUNT(*) FROM marathos_cat.gold.dim_athlete) AS dim_athlete_rows,
    COUNT(DISTINCT event_id) AS unique_events_in_fct,
    (SELECT COUNT(*) FROM marathos_cat.gold.dim_event) AS dim_event_rows,
    COUNT(DISTINCT date_id) AS unique_dates_in_fct,
    (SELECT COUNT(*) FROM marathos_cat.gold.dim_date) AS dim_date_rows
FROM marathos_cat.gold.fct_result;

## Checking unmatched athletes, events, dates

In [0]:
%sql
SELECT COUNT(*) AS unmatched_athletes
FROM marathos_cat.gold.fct_result f
LEFT JOIN marathos_cat.gold.dim_athlete a ON f.athlete_id = a.athlete_id
WHERE a.athlete_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_events
FROM marathos_cat.gold.fct_result f
LEFT JOIN marathos_cat.gold.dim_event e ON f.event_id = e.event_id
WHERE e.event_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_dates
FROM marathos_cat.gold.fct_result f
LEFT JOIN marathos_cat.gold.dim_date d ON f.date_id = d.date_id
WHERE d.date_id IS NULL;

## Checking samples

In [0]:
%sql
--- sample fct_result ---
SELECT * FROM marathos_cat.gold.fct_result LIMIT 5;

In [0]:

%sql
--- sample data dim_athlete ---
SELECT * FROM marathos_cat.gold.dim_athlete LIMIT 5;

In [0]:
%sql
--- sample dim_event ---
SELECT * FROM marathos_cat.gold.dim_event LIMIT 5;

In [0]:
%sql
--- sample dim_date ---
SELECT * FROM marathos_cat.gold.dim_date LIMIT 5;

## Checking mart views

In [0]:
%sql
--- distance by country  ---
SELECT * 
FROM marathos_cat.gold.mart_distance_by_country 
ORDER BY total_races DESC
LIMIT 10;

- France is the most active ultra marathon country
- South Africa are the fastest
- China is the slowest (geography?)



In [0]:
%sql
--- duration_age_gender ---
SELECT * 
FROM marathos_cat.gold.mart_duration_age_gender
ORDER BY avg_run_distance_km DESC
LIMIT 10;

- 100 hours races do not mean more covered/running distance vs 72hours
- Male in their thirties are the most performing age/gender group
- An older group is impressive, the 60-69
- "Short distances" like 48h races have higher avg_speed than 72h ones

In [0]:
%sql
--- top5 duration event  ---
SELECT * FROM marathos_cat.gold.mart_top5_duration_event LIMIT 30;

- Female athletes appear in top 5 races

In [0]:
%sql
--- distance by age and gender  ---
SELECT * FROM marathos_cat.gold.mart_distance_by_age_gender
ORDER BY distance_km DESC
LIMIT 30;

- extreme running distances here and times of course
- Many 0 in avg_finish_hours with nulls for the avg_speed in the same row but others have some speed. POssible abandon or no recoreded data?
- extra filters on the distance could bring more data in
- Some quite old people or high age brackets which is almost suspicious to me but these races cannot be done in 1 go and they need many days, could be a reason for not having a avf_finished_time_hours or avg_speed_km

In [0]:
%sql
--- duration age gender  ---
SELECT * 
FROM marathos_cat.gold.mart_duration_age_gender 
ORDER BY avg_run_distance_km DESC
LIMIT 10;

- 72h races produce the longest distances
- Top rows mostly have total_runners = 1 (?!)
- Shorter races involve more speed (48h races avg_speed > +72h. Tiredness or energy management?
- Males in their thirties are the best performing in duration races

In [0]:
%sql
--- club vs norecorded club runners ---
SELECT * FROM marathos_cat.gold.mart_club_vs_norecordedclub_runners LIMIT 10;


- Club members are faster (both women and men))
- Club members run more races in total
- Runners have almost same average birth year/age in average
- "No club recorded" may mean no club OR missing data — cannot distinguish between the two

In [0]:
%sql
DESCRIBE marathos_cat.gold.fct_result;

In [0]:
%sql
SHOW TABLES IN marathos_cat.gold;

In [0]:
%sql
SELECT
    a.athlete_country,
    ROUND(AVG(f.athlete_average_speed), 2) AS avg_speed_kmh
FROM marathos_cat.gold.fct_result f
LEFT JOIN marathos_cat.gold.dim_athlete a ON f.athlete_id = a.athlete_id
WHERE f.athlete_average_speed IS NOT NULL
    AND a.athlete_country IS NOT NULL
GROUP BY a.athlete_country
ORDER BY avg_speed_kmh DESC
LIMIT 10